# Durable agent workflow with crash-safe state

An agent that takes 8 minutes to complete a multi-step research task crashes 15% of the time mid-task and has to start from scratch. The team is wasting roughly $300 per day in re-executed LLM tokens. You have 120 minutes to build a durable agent using LangGraph checkpoints with a Supabase Postgres checkpointer, then inject three deliberate crashes (process kill via `os.kill`, simulated network blip, simulated LLM timeout) and prove the agent resumes from the last checkpoint within 30 seconds without re-executing already-completed steps.

The killer moment: you kill your own Python process mid-run, the kernel restarts, the agent picks up from the last checkpoint, and the tool-call count in Supabase proves it did not re-execute already-completed steps.

Estimated time: 120 minutes (the killer moment is the kernel-kill recovery; budget time to read the Supabase rows after each recovery). Cost: about $2.00 on a clean run; up to $3.50 if you re-run a task while debugging. Claude Sonnet is the only paid surface; Supabase Postgres holds the checkpoints for free.

Two bucks. The killer moment is when you kill your own kernel.

In [ ]:
# NBVAL_SKIP
# Pinned dependencies. langgraph-checkpoint-postgres is the Postgres-backed
# checkpointer for LangGraph; psycopg2-binary is its driver. labrun-checks
# stays on 0.7.0 per the rest of the track.

!pip install --quiet labrun-checks==0.7.0 langgraph==0.2.74 langgraph-checkpoint-postgres==2.0.13 anthropic==0.42.0 langchain-anthropic==0.3.3 supabase==2.10.0 psycopg2-binary==2.9.10 httpx==0.27.2

In [ ]:
# Imports and per-lab constants. Table identifiers use snake_case under the
# labrun_ prefix so the orphan scan can match by prefix without quoting.

import atexit
import getpass
import json
import os
import signal
import sys
import time
import uuid
from typing import TypedDict, Annotated
from urllib.parse import urlparse

import httpx
import psycopg2

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupAdapter,
    CleanupResource,
)

LAB_ID = "durable-agent-workflow"
LAB_TAG_VALUE = LAB_ID

TABLE_PREFIX = "labrun_durable_agent_workflow_"
CHECKPOINTS_TABLE = TABLE_PREFIX + "checkpoints"
RUNS_TABLE = TABLE_PREFIX + "runs"
TRACE_PATH = "/content/crash_recovery_traces.json"

ANTHROPIC_MODEL = "claude-sonnet-4-5-20250929"
ANTHROPIC_HAIKU_MODEL = "claude-haiku-4-5-20250930"

# Shared thread_id for the killer moment. The whole point of the lab is that
# the same thread_id resumes from the last checkpoint after a crash.
THREAD_ID = f"durable-agent-{uuid.uuid4().hex[:8]}"

In [ ]:
# NBVAL_SKIP
# BYOK setup: session token, ANTHROPIC_API_KEY, SUPABASE_URL,
# SUPABASE_SERVICE_ROLE_KEY. Ping Claude Haiku to validate the Anthropic key
# cheaply, then connect to Supabase Postgres and register the session.

import anthropic
from supabase import create_client

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
ANTHROPIC_API_KEY = getpass.getpass("ANTHROPIC_API_KEY: ").strip()
SUPABASE_URL = getpass.getpass("SUPABASE_URL (https://xxx.supabase.co): ").strip()
SUPABASE_SERVICE_ROLE_KEY = getpass.getpass("SUPABASE_SERVICE_ROLE_KEY: ").strip()

if not all([ANTHROPIC_API_KEY, SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY]):
    print("All three credentials are required. Re-run this cell.")
    raise SystemExit(1)

os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

# Cheap Anthropic ping. One Haiku call: a fraction of a cent.
try:
    _client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    _ping = _client.messages.create(
        model=ANTHROPIC_HAIKU_MODEL,
        max_tokens=8,
        messages=[{"role": "user", "content": "reply with the single word: ok"}],
    )
    print(f"Anthropic credentials validated. Haiku replied: {_ping.content[0].text!r}")
except Exception as exc:
    print(f"Anthropic key rejected: {exc!r}")
    print("Refresh ANTHROPIC_API_KEY and re-run this cell.")
    raise SystemExit(1)

# Supabase REST client (for the runs table) and a direct psycopg2 connection
# string (for the LangGraph PostgresSaver). The service-role key gates the
# REST client; the direct connection uses the Supabase pooled Postgres URL
# that the user pastes from the Supabase dashboard.
supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)

DB_URI = getpass.getpass(
    "Supabase Postgres connection string "
    "(Project Settings > Database > Connection string > URI; "
    "use the Transaction pooler or Session pooler URL): "
).strip()

try:
    _conn = psycopg2.connect(DB_URI)
    _conn.autocommit = True
    with _conn.cursor() as _cur:
        _cur.execute("SELECT 1")
    _conn.close()
    print("Supabase Postgres connection validated.")
except Exception as exc:
    print(f"Postgres connection failed: {exc!r}")
    print("Re-check the connection string (must include the password, port 6543 pooler is recommended).")
    raise SystemExit(1)

SUPABASE_CREDENTIALS = {
    "supabase_url": SUPABASE_URL,
    "service_role_key": SUPABASE_SERVICE_ROLE_KEY,
    "db_uri": DB_URI,
}

register(session_token=session_token, lab_id=LAB_ID, credentials=SUPABASE_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")
print(f"Thread id for this session: {THREAD_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, custom Supabase adapter, atexit safety net, orphan scan.
# Manifest is module-level. The adapter drops Supabase tables by exact name.
# TODO: migrate to vector_db.py once it supports arbitrary table cleanup by prefix.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="supabase_table",
        id=RUNS_TABLE,
        region="supabase",
        cli_delete_command=f"psql $DB_URI -c 'DROP TABLE IF EXISTS public.{RUNS_TABLE} CASCADE'",
    ),
    CleanupResource(
        type="supabase_table",
        id=CHECKPOINTS_TABLE,
        region="supabase",
        cli_delete_command=f"psql $DB_URI -c 'DROP TABLE IF EXISTS public.{CHECKPOINTS_TABLE} CASCADE'",
    ),
    CleanupResource(
        type="langgraph_checkpointer_tables",
        id="langgraph_checkpointer_schema",
        region="supabase",
        cli_delete_command="psql $DB_URI -c 'DROP TABLE IF EXISTS checkpoints, checkpoint_blobs, checkpoint_writes, checkpoint_migrations CASCADE'",
    ),
    CleanupResource(
        type="local_file",
        id=TRACE_PATH,
        region="local",
        cli_delete_command=f"rm -f {TRACE_PATH}",
    ),
]


class DurableAgentCleanupAdapter:
    """Drops Supabase tables by exact name and removes the local trace file.

    TODO: migrate to vector_db.py once it supports arbitrary table cleanup
    by prefix. For now the adapter is inline so the lab is self-contained.
    """

    def __init__(self, db_uri):
        self._db_uri = db_uri

    def _exec(self, sql):
        conn = psycopg2.connect(self._db_uri)
        conn.autocommit = True
        try:
            with conn.cursor() as cur:
                cur.execute(sql)
        finally:
            conn.close()

    def _table_exists(self, table_name):
        conn = psycopg2.connect(self._db_uri)
        try:
            with conn.cursor() as cur:
                cur.execute(
                    "SELECT 1 FROM information_schema.tables "
                    "WHERE table_schema = 'public' AND table_name = %s",
                    (table_name,),
                )
                return cur.fetchone() is not None
        finally:
            conn.close()

    def delete_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "supabase_table":
            self._exec(f"DROP TABLE IF EXISTS public.{rid} CASCADE")
        elif rtype == "langgraph_checkpointer_tables":
            for t in ("checkpoint_writes", "checkpoint_blobs", "checkpoints", "checkpoint_migrations"):
                self._exec(f"DROP TABLE IF EXISTS public.{t} CASCADE")
        elif rtype == "local_file":
            try:
                os.remove(rid)
            except FileNotFoundError:
                pass
        else:
            raise ValueError(f"DurableAgentCleanupAdapter: unknown resource type {rtype!r}")

    def describe_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "supabase_table":
            try:
                return self._table_exists(rid)
            except Exception:
                return False
        if rtype == "langgraph_checkpointer_tables":
            try:
                for t in ("checkpoints", "checkpoint_blobs", "checkpoint_writes", "checkpoint_migrations"):
                    if self._table_exists(t):
                        return True
                return False
            except Exception:
                return False
        if rtype == "local_file":
            return os.path.exists(rid)
        return False

    def tag_scan(self, credentials, lab_slug, region):
        # Supabase has no tag-resources API. The orphan scan instead looks for
        # any table in the public schema whose name starts with the lab's
        # table prefix, returning the bare table name as the "arn".
        try:
            conn = psycopg2.connect(self._db_uri)
            try:
                with conn.cursor() as cur:
                    cur.execute(
                        "SELECT table_name FROM information_schema.tables "
                        "WHERE table_schema = 'public' AND table_name LIKE %s",
                        (TABLE_PREFIX + "%",),
                    )
                    return [row[0] for row in cur.fetchall()]
            finally:
                conn.close()
        except Exception:
            return []


CLEANUP_ADAPTER = DurableAgentCleanupAdapter(db_uri=DB_URI)


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans():
    return CLEANUP_ADAPTER.tag_scan(SUPABASE_CREDENTIALS, LAB_TAG_VALUE, "supabase")


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing Supabase tables with prefix {TABLE_PREFIX!r} were found:")
    for name in orphans:
        print("  -", name)
    print()
    print("Run the cleanup cell at the bottom first, then re-run setup.")
    raise SystemExit(1)

# Seed the runs table the lab uses for its bookkeeping. The LangGraph
# checkpointer tables get created later by PostgresSaver.setup().
_seed_conn = psycopg2.connect(DB_URI)
_seed_conn.autocommit = True
try:
    with _seed_conn.cursor() as _cur:
        _cur.execute(
            f"""
            CREATE TABLE IF NOT EXISTS public.{RUNS_TABLE} (
                id bigserial PRIMARY KEY,
                thread_id text NOT NULL,
                run_label text NOT NULL,
                step_name text NOT NULL,
                tool_calls_in_step integer NOT NULL DEFAULT 0,
                created_at timestamptz NOT NULL DEFAULT now()
            )
            """
        )
finally:
    _seed_conn.close()

print("No prior orphans found. Safe to create resources for this session.")
print(f"Bookkeeping table {RUNS_TABLE} ready.")

## Build the durable LangGraph agent

The agent answers a research question by walking three tools: `search(query)` returns matching doc IDs from a small synthetic knowledge base, `read_doc(doc_id)` returns the full text of one doc, and `summarize(text)` asks Claude to compress the text into one sentence. The agent finishes by composing a citation-formatted answer from the summaries.

Two design choices the brief calls out:

1. Each LangGraph node returns a **partial state update**, not the full state. Returning the full state is the most common bug here; it overwrites the checkpointer's accumulated state and the resume looks like a fresh start.
2. Retries live **inside the node**, not at the graph level. A graph-level retry restarts the entire workflow on a transient error and burns 5x the tokens. A node-level retry catches the transient error, re-runs only that node, and the checkpointer keeps everything before it intact.

Knowledge base: 5 inline docs about durable agent patterns. Seeded once at setup.

In [ ]:
# Graph definition (Level 2 skeleton). The state, the tool implementations,
# and the trace helpers are wired up; the student writes the node bodies and
# the graph assembly.

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.postgres import PostgresSaver

# Synthetic knowledge base. Five docs, each tied to one durable-agent concept.
KNOWLEDGE_BASE = {
    "doc-001": (
        "Title: Durable execution. Durable execution guarantees that a workflow's "
        "progress survives process crashes by persisting state after each step. "
        "Temporal, Inngest, and LangGraph all implement this pattern."
    ),
    "doc-002": (
        "Title: Checkpointers. A checkpointer writes the graph state to a backing "
        "store after every node. LangGraph ships in-memory, SQLite, and Postgres "
        "checkpointers; production uses Postgres for the shared, persistent option."
    ),
    "doc-003": (
        "Title: Thread IDs. A thread_id is the identity of a single agent run "
        "across crashes and resumes. Invoking the graph with the same thread_id "
        "after a crash resumes from the last checkpoint; a new thread_id starts fresh."
    ),
    "doc-004": (
        "Title: Idempotent tools. Tool implementations should be idempotent so that "
        "at-least-once resume semantics from the checkpointer compose into "
        "effectively-exactly-once end-to-end semantics for the workflow."
    ),
    "doc-005": (
        "Title: Retry boundaries. Retries belong inside the node, not at the graph "
        "level. A graph-level retry restarts the whole workflow on a transient "
        "error; a node-level retry preserves the checkpointer's accumulated state."
    ),
}

RESEARCH_QUESTION = (
    "How does LangGraph make agent workflows survive crashes, and what "
    "design choices matter for production?"
)

TOOL_CALL_COUNTER = {"count": 0}
TRACE = []


def _record_step(thread_id, run_label, step_name, tool_calls):
    """Insert a row into the runs table and append to the local trace."""
    TRACE.append({
        "thread_id": thread_id,
        "run_label": run_label,
        "step_name": step_name,
        "tool_calls_in_step": tool_calls,
        "ts": time.time(),
    })
    with open(TRACE_PATH, "w") as f:
        json.dump(TRACE, f, indent=2)
    supabase.table(RUNS_TABLE).insert({
        "thread_id": thread_id,
        "run_label": run_label,
        "step_name": step_name,
        "tool_calls_in_step": tool_calls,
    }).execute()


# State schema for the graph. Annotated reducers tell LangGraph how to merge
# partial updates from each node.
def _append(left, right):
    return (left or []) + (right or [])


class AgentState(TypedDict):
    question: str
    doc_ids: Annotated[list, _append]
    docs: Annotated[list, _append]
    summaries: Annotated[list, _append]
    answer: str
    run_label: str


# Inline crash-injection helper. Reads an env var and raises SystemExit at the
# specified step. SystemExit propagates out of the graph and crashes the process
# in a way that the LangGraph checkpointer treats as a clean abort: the last
# successful node's state is durable in Postgres.
def inject_crash_at_step_n(n):
    """Tell the agent to call os.kill on itself when it enters step n."""
    os.environ["LABRUN_CRASH_AT_STEP"] = str(n)


def clear_crash_injection():
    os.environ.pop("LABRUN_CRASH_AT_STEP", None)


STEP_INDEX = {"search": 1, "read_1": 2, "read_2": 3, "summarize_1": 4, "summarize_2": 5, "compose": 6}


def _maybe_crash(step_name):
    target = os.environ.get("LABRUN_CRASH_AT_STEP")
    if target is None:
        return
    if int(target) == STEP_INDEX.get(step_name, -1):
        print(f"\n>>> CRASH INJECTED at step {target} ({step_name}). Sending SIGTERM to self. <<<\n")
        os.environ.pop("LABRUN_CRASH_AT_STEP", None)
        os.kill(os.getpid(), signal.SIGTERM)


# Tool implementations. Each one bumps the global TOOL_CALL_COUNTER, which the
# checkpoint validator reads to confirm no re-execution happened.
def tool_search(query):
    TOOL_CALL_COUNTER["count"] += 1
    matches = []
    qlow = query.lower()
    for doc_id, text in KNOWLEDGE_BASE.items():
        if any(word in text.lower() for word in qlow.split() if len(word) > 4):
            matches.append(doc_id)
    return matches[:3] if matches else list(KNOWLEDGE_BASE.keys())[:3]


_NETWORK_BLIP_FIRED = {"yes": False}


def tool_read_doc(doc_id):
    TOOL_CALL_COUNTER["count"] += 1
    if os.environ.get("LABRUN_NETWORK_BLIP") == "1" and not _NETWORK_BLIP_FIRED["yes"]:
        _NETWORK_BLIP_FIRED["yes"] = True
        raise httpx.ConnectError("simulated network blip")
    return KNOWLEDGE_BASE.get(doc_id, "(doc not found)")


_anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)


def tool_summarize(text):
    TOOL_CALL_COUNTER["count"] += 1
    msg = _anthropic_client.messages.create(
        model=ANTHROPIC_HAIKU_MODEL,
        max_tokens=120,
        messages=[{
            "role": "user",
            "content": f"Summarize this in one sentence, keep the title:\n\n{text}",
        }],
    )
    return msg.content[0].text


# Node bodies. Each returns a PARTIAL state update. Returning the full state
# is the most common bug; it overwrites the checkpointer's accumulated state.
def node_search(state):
    _maybe_crash("search")
    ids = tool_search(state["question"])
    _record_step(THREAD_ID, state.get("run_label", "unknown"), "search", 1)
    # YOUR CODE: return a partial state update with the doc_ids field set to ids
    # YOUR CODE: do NOT return the full state; the reducer in AgentState merges this


def node_read_1(state):
    _maybe_crash("read_1")
    doc = tool_read_doc(state["doc_ids"][0])
    _record_step(THREAD_ID, state.get("run_label", "unknown"), "read_1", 1)
    # YOUR CODE: return a partial state update appending [doc] to docs


def node_read_2(state):
    _maybe_crash("read_2")
    second_id = state["doc_ids"][1] if len(state["doc_ids"]) > 1 else state["doc_ids"][0]
    doc = tool_read_doc(second_id)
    _record_step(THREAD_ID, state.get("run_label", "unknown"), "read_2", 1)
    # YOUR CODE: return a partial state update appending [doc] to docs


def node_summarize_1(state):
    _maybe_crash("summarize_1")
    summary = _llm_call_with_retry(state["docs"][0])
    _record_step(THREAD_ID, state.get("run_label", "unknown"), "summarize_1", 1)
    # YOUR CODE: return a partial state update appending [summary] to summaries


def node_summarize_2(state):
    _maybe_crash("summarize_2")
    summary = _llm_call_with_retry(state["docs"][1])
    _record_step(THREAD_ID, state.get("run_label", "unknown"), "summarize_2", 1)
    # YOUR CODE: return a partial state update appending [summary] to summaries


def node_compose(state):
    _maybe_crash("compose")
    citations = ", ".join(state["doc_ids"])
    bullets = "\n".join(f"- {s}" for s in state["summaries"])
    answer = (
        f"Question: {state['question']}\n\nFindings (citing {citations}):\n{bullets}"
    )
    _record_step(THREAD_ID, state.get("run_label", "unknown"), "compose", 0)
    # YOUR CODE: return a partial state update with the answer field set to answer


# CP3 + CP4 retry helper. Lives INSIDE the LLM-call node. A graph-level retry
# would re-execute every prior node and burn tokens.
def _llm_call_with_retry(doc_text, max_attempts=3):
    last_exc = None
    for attempt in range(max_attempts):
        try:
            return tool_summarize(doc_text)
        except (httpx.ConnectError, anthropic.APITimeoutError) as exc:
            last_exc = exc
            print(f"    transient error on attempt {attempt + 1}: {exc!r}; retrying inside node")
            time.sleep(1)
    raise RuntimeError(f"LLM call failed after {max_attempts} attempts: {last_exc!r}")


# YOUR CODE: build the StateGraph: add the six nodes by name, set the entry
# YOUR CODE: point to 'search', add edges search -> read_1 -> read_2 ->
# YOUR CODE: summarize_1 -> summarize_2 -> compose -> END, then compile() the
# YOUR CODE: graph against a PostgresSaver checkpointer connected to DB_URI.
# YOUR CODE: Call saver.setup() before compile so LangGraph's tables exist.
# YOUR CODE: Assign the compiled graph to the variable name `graph`.

graph = None  # student replaces this

<details><summary>Hint 1 (nudge)</summary>

Each node's return value is merged into the running state by the reducer declared on the `AgentState` TypedDict. Return only the fields the node produced, not the entire state.

For the graph itself: `PostgresSaver.from_conn_string(DB_URI)` is a context manager in some versions of `langgraph-checkpoint-postgres`; in 2.0.x you can also construct it from a `Connection` you create yourself. Either way, call `saver.setup()` once before `compile(checkpointer=saver)`.

</details>

<details><summary>Hint 2 (direction)</summary>

Node return shape: `return {"doc_ids": ids}` (NOT `return {**state, "doc_ids": ids}`).

Graph wiring:

```
builder = StateGraph(AgentState)
builder.add_node("search", node_search)
builder.add_node("read_1", node_read_1)
...
builder.set_entry_point("search")
builder.add_edge("search", "read_1")
...
builder.add_edge("compose", END)
```

Compile against a `PostgresSaver` constructed from `DB_URI`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Full working node bodies and graph assembly:

```python
def node_search(state):
    _maybe_crash("search")
    ids = tool_search(state["question"])
    _record_step(THREAD_ID, state.get("run_label", "unknown"), "search", 1)
    return {"doc_ids": ids}

def node_read_1(state):
    _maybe_crash("read_1")
    doc = tool_read_doc(state["doc_ids"][0])
    _record_step(THREAD_ID, state.get("run_label", "unknown"), "read_1", 1)
    return {"docs": [doc]}

def node_read_2(state):
    _maybe_crash("read_2")
    second_id = state["doc_ids"][1] if len(state["doc_ids"]) > 1 else state["doc_ids"][0]
    doc = tool_read_doc(second_id)
    _record_step(THREAD_ID, state.get("run_label", "unknown"), "read_2", 1)
    return {"docs": [doc]}

def node_summarize_1(state):
    _maybe_crash("summarize_1")
    summary = _llm_call_with_retry(state["docs"][0])
    _record_step(THREAD_ID, state.get("run_label", "unknown"), "summarize_1", 1)
    return {"summaries": [summary]}

def node_summarize_2(state):
    _maybe_crash("summarize_2")
    summary = _llm_call_with_retry(state["docs"][1])
    _record_step(THREAD_ID, state.get("run_label", "unknown"), "summarize_2", 1)
    return {"summaries": [summary]}

def node_compose(state):
    _maybe_crash("compose")
    citations = ", ".join(state["doc_ids"])
    bullets = "\n".join(f"- {s}" for s in state["summaries"])
    answer = f"Question: {state['question']}\n\nFindings (citing {citations}):\n{bullets}"
    _record_step(THREAD_ID, state.get("run_label", "unknown"), "compose", 0)
    return {"answer": answer}

saver = PostgresSaver.from_conn_string(DB_URI).__enter__()
saver.setup()

builder = StateGraph(AgentState)
for name, fn in [
    ("search", node_search), ("read_1", node_read_1), ("read_2", node_read_2),
    ("summarize_1", node_summarize_1), ("summarize_2", node_summarize_2),
    ("compose", node_compose),
]:
    builder.add_node(name, fn)
builder.set_entry_point("search")
builder.add_edge("search", "read_1")
builder.add_edge("read_1", "read_2")
builder.add_edge("read_2", "summarize_1")
builder.add_edge("summarize_1", "summarize_2")
builder.add_edge("summarize_2", "compose")
builder.add_edge("compose", END)

graph = builder.compile(checkpointer=saver)
```

</details>

## Task 1: Clean run end-to-end

Invoke the graph once with no crash injection. The agent should run six nodes, generate one answer with two cited sources, and write at least five rows into the runs table for `thread_id=THREAD_ID, run_label='clean'`. The shared checkpoint counter at the bottom of the run is the proof artifact for this task.

In [ ]:
# NBVAL_SKIP
# Task 1: invoke the graph for a clean run, no crash injection.

clear_crash_injection()
TOOL_CALL_COUNTER["count"] = 0

# YOUR CODE: invoke graph with initial state
# YOUR CODE:   {"question": RESEARCH_QUESTION, "doc_ids": [], "docs": [],
# YOUR CODE:    "summaries": [], "answer": "", "run_label": "clean"}
# YOUR CODE: pass config={"configurable": {"thread_id": THREAD_ID}}
# YOUR CODE: assign the result to `clean_result`
# YOUR CODE: print clean_result["answer"] so the operator can read it

clean_result = None

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: the runs table has >= 5 tool calls for the clean run on this thread.


def checkpoint_1(session):
    try:
        resp = (
            supabase.table(RUNS_TABLE)
            .select("step_name,tool_calls_in_step")
            .eq("thread_id", THREAD_ID)
            .eq("run_label", "clean")
            .execute()
        )
    except Exception as exc:
        return CheckpointResult(
            status="error",
            error_reason=f"Could not query {RUNS_TABLE}: {exc!r}",
        )
    rows = resp.data or []
    total_tool_calls = sum(r.get("tool_calls_in_step", 0) for r in rows)
    if total_tool_calls < 5:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Clean run recorded total_tool_calls={total_tool_calls} for thread_id={THREAD_ID!r}; "
                f"expected >= 5. The most common cause is a node returning the full state instead of "
                f"a partial update, which makes earlier writes look like they never happened."
            ),
        )
    if not isinstance(clean_result, dict) or not clean_result.get("answer"):
        return CheckpointResult(
            status="fail",
            error_reason="clean_result.answer is empty; the compose node did not produce an answer.",
        )
    return CheckpointResult(status="pass")


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The graph wants the initial state to include every field declared on `AgentState`, with the list fields empty and the string fields empty. The config dict carries the `thread_id` under `configurable`.

</details>

<details><summary>Hint 2 (direction)</summary>

```
initial = {
    "question": RESEARCH_QUESTION,
    "doc_ids": [], "docs": [], "summaries": [], "answer": "",
    "run_label": "clean",
}
config = {"configurable": {"thread_id": THREAD_ID}}
clean_result = graph.invoke(initial, config=config)
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1:

```python
initial = {
    "question": RESEARCH_QUESTION,
    "doc_ids": [],
    "docs": [],
    "summaries": [],
    "answer": "",
    "run_label": "clean",
}
config = {"configurable": {"thread_id": THREAD_ID}}
clean_result = graph.invoke(initial, config=config)
print(clean_result["answer"])
```

</details>

**Wallet check.** Clean run cost: two Haiku summary calls plus a handful of cheap tool calls. Roughly 8000 tokens billed. About $0.20 spent. Your morning coffee cost 25x more.

## Task 2: The killer moment, deliberate kernel kill at step 3

Call `inject_crash_at_step_n(3)` and re-invoke the graph with a new `run_label='crash_step_3'` but the SAME `thread_id`. The agent will reach the `read_2` node (step 3), the injected crash will `os.kill` the Python process, and the Colab kernel will need to be restarted.

After the kernel restart you will need to re-import everything and re-establish the connection. The single most important rule: **use the same `THREAD_ID`**. The checkpointer keyed all the saved state on that string; a new thread_id starts fresh and burns tokens.

Validator: the runs table for this thread under `run_label='crash_step_3'` should NOT contain `search` or `read_1` rows (those steps were already durable before the crash). The post-crash run should contain `read_2`, `summarize_1`, `summarize_2`, and `compose`.

In [ ]:
# NBVAL_SKIP
# Task 2: inject the crash, run the graph, watch the kernel die. Then:
# 1. Use Colab > Runtime > Restart runtime.
# 2. Re-run every cell above this one (setup, manifest, graph definition).
#    Critical: do NOT regenerate THREAD_ID. Set it back to the value the setup
#    cell printed before you crashed.
# 3. Then run the resume invocation below.

inject_crash_at_step_n(3)

config = {"configurable": {"thread_id": THREAD_ID}}
crash_initial = {
    "question": RESEARCH_QUESTION,
    "doc_ids": [],
    "docs": [],
    "summaries": [],
    "answer": "",
    "run_label": "crash_step_3",
}

# YOUR CODE: call graph.invoke(crash_initial, config=config) inside a
# YOUR CODE: try-except SystemExit so the cell does not silently exit.
# YOUR CODE: print a clear "crash injected" message if SystemExit fires.
# YOUR CODE: After the kernel restart, set THREAD_ID back to its prior value
# YOUR CODE:   (the setup cell printed it; copy that exact string here BEFORE
# YOUR CODE:   re-running setup so the regenerated THREAD_ID is overwritten):
# YOUR CODE: # THREAD_ID = "durable-agent-XXXXXXXX"  # paste your saved value

In [ ]:
# NBVAL_SKIP
# Resume invocation. Run this AFTER the kernel has been restarted and the
# graph has been rebuilt with the same THREAD_ID. The PostgresSaver looks up
# the last successful checkpoint by thread_id and resumes from there.

clear_crash_injection()
config = {"configurable": {"thread_id": THREAD_ID}}

# YOUR CODE: graph.invoke(None, config=config) resumes from the last
# YOUR CODE:   checkpoint without re-running prior nodes. Passing None as
# YOUR CODE:   the input is the LangGraph contract for resume.
# YOUR CODE: assign the result to `resumed_result` and print resumed_result["answer"].

resumed_result = None

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: the crash run did NOT re-execute steps 1-2 (search, read_1).
# The runs table for this thread under run_label='crash_step_3' should show
# only the post-crash steps that resumed after the kernel restart.


def checkpoint_2(session):
    try:
        resp = (
            supabase.table(RUNS_TABLE)
            .select("step_name")
            .eq("thread_id", THREAD_ID)
            .eq("run_label", "crash_step_3")
            .execute()
        )
    except Exception as exc:
        return CheckpointResult(
            status="error",
            error_reason=f"Could not query {RUNS_TABLE}: {exc!r}",
        )
    crash_steps = {r["step_name"] for r in (resp.data or [])}
    re_executed = crash_steps & {"search", "read_1"}
    if re_executed:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Crash run re-executed steps {sorted(re_executed)}. The checkpointer should have skipped "
                f"them. Most common cause: thread_id changed between the crashing run and the resume run. "
                f"Confirm THREAD_ID has the same value both times."
            ),
        )
    expected_post_crash = {"read_2", "summarize_1", "summarize_2", "compose"}
    missing = expected_post_crash - crash_steps
    if missing:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Resume did not complete: missing post-crash steps {sorted(missing)}. "
                f"Did you run the resume cell with graph.invoke(None, config=config) after the kernel restart?"
            ),
        )
    return CheckpointResult(status="pass")


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

The crash sends `SIGTERM` to the Python process. The cell will exit. Wrap the invoke in `try/except SystemExit` so you can print a clear marker before the kernel dies, then follow the runtime-restart steps in the task markdown.

On resume: the LangGraph contract is `graph.invoke(None, config={"configurable": {"thread_id": <same id>}})`. Passing `None` as the input is the signal to resume.

</details>

<details><summary>Hint 2 (direction)</summary>

```
try:
    graph.invoke(crash_initial, config=config)
except SystemExit:
    print("Crash fired. Restart the runtime now (Runtime > Restart runtime),")
    print("re-run setup with the SAME THREAD_ID, then run the resume cell.")
```

After kernel restart, before re-running setup, you must edit the constants cell to pin THREAD_ID to its prior value. Then the resume cell does `graph.invoke(None, config={"configurable": {"thread_id": THREAD_ID}})`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2:

Crash cell:

```python
try:
    graph.invoke(crash_initial, config=config)
except SystemExit:
    print("\nCrash injected at step 3. The Python process is about to exit.")
    print("Next: Runtime > Restart runtime, then re-run setup with the same THREAD_ID.")
```

Resume cell (after kernel restart and setup re-run with pinned THREAD_ID):

```python
clear_crash_injection()
config = {"configurable": {"thread_id": THREAD_ID}}
resumed_result = graph.invoke(None, config=config)
print(resumed_result["answer"])
```

</details>

**Wallet check.** Crash 1 of 3 recovered. Tool calls saved by the checkpointer: 3 (search + read_1, plus the partial step 3 work). Spent so far: about $0.30 on Claude. The naive non-durable agent would have re-run the entire workflow and paid for those three steps twice.

## Task 3: Simulated network blip during a tool call

Set the `LABRUN_NETWORK_BLIP=1` env var and re-invoke the graph with a NEW thread_id (this is a fresh end-to-end run, not a resume). The `tool_read_doc` function raises `httpx.ConnectError` exactly once on its first call. The retry must happen inside the node, not at the graph level. The graph-level retry pattern would re-run `search` and burn tokens; the node-level retry pattern catches the error inside `tool_read_doc`'s caller, retries the single call, and the checkpointer keeps the earlier `search` state intact.

In [ ]:
# NBVAL_SKIP
# Task 3: enable the network blip, run with a fresh thread, watch the retry.
# Wrap tool_read_doc with a small retry helper so the read node retries
# inside the node (not at the graph level).

BLIP_THREAD_ID = f"durable-agent-blip-{uuid.uuid4().hex[:8]}"
os.environ["LABRUN_NETWORK_BLIP"] = "1"
_NETWORK_BLIP_FIRED["yes"] = False  # reset the fire-once latch for this run

# YOUR CODE: define _read_with_retry(doc_id) that calls tool_read_doc with
# YOUR CODE:   try-except httpx.ConnectError, up to 3 attempts.
# YOUR CODE: define node_read_1_retry and node_read_2_retry that record under
# YOUR CODE:   BLIP_THREAD_ID and call _read_with_retry.
# YOUR CODE: build a fresh StateGraph with these retry-aware read nodes and
# YOUR CODE:   compile it against a PostgresSaver. Name it blip_graph.
# YOUR CODE: invoke blip_graph with run_label='blip' and BLIP_THREAD_ID.
# YOUR CODE:   Assign the result to `blip_result`.
# YOUR CODE: clean up: os.environ.pop("LABRUN_NETWORK_BLIP", None)

blip_result = None

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: the blip run completed; `search` ran exactly once (the
# node-level retry did not re-run prior nodes).


def checkpoint_3(session):
    try:
        resp = (
            supabase.table(RUNS_TABLE)
            .select("step_name")
            .eq("thread_id", BLIP_THREAD_ID)
            .eq("run_label", "blip")
            .execute()
        )
    except Exception as exc:
        return CheckpointResult(
            status="error",
            error_reason=f"Could not query {RUNS_TABLE}: {exc!r}",
        )
    rows = resp.data or []
    step_names = [r["step_name"] for r in rows]
    search_count = step_names.count("search")
    if search_count == 0:
        return CheckpointResult(
            status="fail",
            error_reason=(
                "Blip run produced zero search-step rows for the new thread_id. Did the graph invoke?"
            ),
        )
    if search_count > 1:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Search step ran {search_count} times on the blip run. "
                f"The retry is being done at the graph level. Move it inside the read node "
                f"so only the failing call retries."
            ),
        )
    if "compose" not in step_names:
        return CheckpointResult(
            status="fail",
            error_reason=(
                "Blip run never reached compose; the retry did not recover the read call."
            ),
        )
    return CheckpointResult(status="pass")


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

The retry has to wrap the single failing call, not the surrounding LangGraph node body and not the graph invoke. The `_llm_call_with_retry` helper at the top of the graph cell is the right shape; write the equivalent for `tool_read_doc`.

</details>

<details><summary>Hint 2 (direction)</summary>

Define a wrapper:

```
def _read_with_retry(doc_id, max_attempts=3):
    for attempt in range(max_attempts):
        try:
            return tool_read_doc(doc_id)
        except httpx.ConnectError as exc:
            print(f"  network blip on attempt {attempt + 1}: {exc!r}; retrying inside node")
            time.sleep(1)
    raise RuntimeError("read_doc failed after retries")
```

Then build a fresh graph whose `read_1` and `read_2` nodes call `_read_with_retry` and record under `BLIP_THREAD_ID`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3:

```python
BLIP_THREAD_ID = f"durable-agent-blip-{uuid.uuid4().hex[:8]}"
os.environ["LABRUN_NETWORK_BLIP"] = "1"
_NETWORK_BLIP_FIRED["yes"] = False

def _read_with_retry(doc_id, max_attempts=3):
    for attempt in range(max_attempts):
        try:
            return tool_read_doc(doc_id)
        except httpx.ConnectError as exc:
            print(f"  network blip on attempt {attempt + 1}: {exc!r}; retrying inside node")
            time.sleep(1)
    raise RuntimeError("read_doc failed after retries")

def node_search_blip(state):
    ids = tool_search(state["question"])
    _record_step(BLIP_THREAD_ID, state.get("run_label", "unknown"), "search", 1)
    return {"doc_ids": ids}

def node_read_1_retry(state):
    doc = _read_with_retry(state["doc_ids"][0])
    _record_step(BLIP_THREAD_ID, state.get("run_label", "unknown"), "read_1", 1)
    return {"docs": [doc]}

def node_read_2_retry(state):
    second_id = state["doc_ids"][1] if len(state["doc_ids"]) > 1 else state["doc_ids"][0]
    doc = _read_with_retry(second_id)
    _record_step(BLIP_THREAD_ID, state.get("run_label", "unknown"), "read_2", 1)
    return {"docs": [doc]}

def node_summarize_1_blip(state):
    summary = _llm_call_with_retry(state["docs"][0])
    _record_step(BLIP_THREAD_ID, state.get("run_label", "unknown"), "summarize_1", 1)
    return {"summaries": [summary]}

def node_summarize_2_blip(state):
    summary = _llm_call_with_retry(state["docs"][1])
    _record_step(BLIP_THREAD_ID, state.get("run_label", "unknown"), "summarize_2", 1)
    return {"summaries": [summary]}

def node_compose_blip(state):
    citations = ", ".join(state["doc_ids"])
    bullets = "\n".join(f"- {s}" for s in state["summaries"])
    answer = f"Question: {state['question']}\n\nFindings (citing {citations}):\n{bullets}"
    _record_step(BLIP_THREAD_ID, state.get("run_label", "unknown"), "compose", 0)
    return {"answer": answer}

saver = PostgresSaver.from_conn_string(DB_URI).__enter__()
saver.setup()
builder = StateGraph(AgentState)
for name, fn in [
    ("search", node_search_blip), ("read_1", node_read_1_retry), ("read_2", node_read_2_retry),
    ("summarize_1", node_summarize_1_blip), ("summarize_2", node_summarize_2_blip),
    ("compose", node_compose_blip),
]:
    builder.add_node(name, fn)
builder.set_entry_point("search")
builder.add_edge("search", "read_1")
builder.add_edge("read_1", "read_2")
builder.add_edge("read_2", "summarize_1")
builder.add_edge("summarize_1", "summarize_2")
builder.add_edge("summarize_2", "compose")
builder.add_edge("compose", END)
blip_graph = builder.compile(checkpointer=saver)

initial = {
    "question": RESEARCH_QUESTION,
    "doc_ids": [], "docs": [], "summaries": [], "answer": "",
    "run_label": "blip",
}
blip_result = blip_graph.invoke(initial, config={"configurable": {"thread_id": BLIP_THREAD_ID}})
os.environ.pop("LABRUN_NETWORK_BLIP", None)
print(blip_result["answer"])
```

</details>

**Wallet check.** Crash 2 of 3 recovered. Tool calls saved: 1 (the inner retry caught the blip; the prior node's state was preserved). Spent so far: about $0.60 on Claude across two end-to-end runs and one crash-recovery.

## Task 4: Simulated LLM timeout

Patch `_anthropic_client.messages.create` to raise `anthropic.APITimeoutError` once at the first summary call, then succeed. Same shape as Task 3 but for the LLM-call node. The `_llm_call_with_retry` helper that the graph definition already wires into `node_summarize_1` and `node_summarize_2` is what catches this; the student verifies the pattern works for LLM timeouts as well as network errors.

In [ ]:
# NBVAL_SKIP
# Task 4: monkey-patch the Anthropic client to raise APITimeoutError once,
# then run with a NEW thread_id.

TIMEOUT_THREAD_ID = f"durable-agent-timeout-{uuid.uuid4().hex[:8]}"

_timeout_fired = {"yes": False}
_real_messages_create = _anthropic_client.messages.create


def _patched_messages_create(*args, **kwargs):
    if not _timeout_fired["yes"]:
        _timeout_fired["yes"] = True
        try:
            raise anthropic.APITimeoutError(
                request=httpx.Request("POST", "https://api.anthropic.com/v1/messages")
            )
        except TypeError:
            raise anthropic.APITimeoutError("simulated LLM timeout")
    return _real_messages_create(*args, **kwargs)


# YOUR CODE: monkey-patch _anthropic_client.messages.create with _patched_messages_create
# YOUR CODE: invoke graph with a fresh initial state and TIMEOUT_THREAD_ID,
# YOUR CODE:   run_label='timeout'. Note that the existing graph closes over
# YOUR CODE:   THREAD_ID for its _record_step calls, so for this task you may
# YOUR CODE:   rebuild a small graph whose nodes record under TIMEOUT_THREAD_ID,
# YOUR CODE:   OR re-assign THREAD_ID = TIMEOUT_THREAD_ID before invoke. Either
# YOUR CODE:   is acceptable; the checkpoint just reads rows from the runs table.
# YOUR CODE: Assign the result to `timeout_result`.
# YOUR CODE: restore the original after the run:
# YOUR CODE:   _anthropic_client.messages.create = _real_messages_create

timeout_result = None

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: the timeout run completed and the planner ran only once
# (proving the retry was inside the LLM-call node, not at the graph level).


def checkpoint_4(session):
    try:
        resp = (
            supabase.table(RUNS_TABLE)
            .select("step_name")
            .eq("thread_id", TIMEOUT_THREAD_ID)
            .eq("run_label", "timeout")
            .execute()
        )
    except Exception as exc:
        return CheckpointResult(
            status="error",
            error_reason=f"Could not query {RUNS_TABLE}: {exc!r}",
        )
    rows = resp.data or []
    step_names = [r["step_name"] for r in rows]
    search_count = step_names.count("search")
    if search_count == 0:
        return CheckpointResult(
            status="fail",
            error_reason="timeout run produced no rows for the new thread_id; did the graph invoke?",
        )
    if search_count > 1:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Search step ran {search_count} times under the timeout run. "
                f"The retry is at the graph level. Move it inside the LLM-call node."
            ),
        )
    if "compose" not in step_names:
        return CheckpointResult(
            status="fail",
            error_reason="timeout run never reached compose; the retry did not recover the LLM call.",
        )
    return CheckpointResult(status="pass")


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

The `_llm_call_with_retry` helper inside `tool_summarize`'s caller node already catches `anthropic.APITimeoutError`. The work for this task is to wire up the monkey-patch and invoke the graph; no new retry logic is needed.

</details>

<details><summary>Hint 2 (direction)</summary>

```
_anthropic_client.messages.create = _patched_messages_create
THREAD_ID_BACKUP = THREAD_ID
THREAD_ID = TIMEOUT_THREAD_ID  # so _record_step writes under the timeout thread
initial = {"question": RESEARCH_QUESTION, "doc_ids": [], "docs": [], "summaries": [], "answer": "", "run_label": "timeout"}
timeout_result = graph.invoke(initial, config={"configurable": {"thread_id": TIMEOUT_THREAD_ID}})
_anthropic_client.messages.create = _real_messages_create
THREAD_ID = THREAD_ID_BACKUP
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4:

```python
_anthropic_client.messages.create = _patched_messages_create
THREAD_ID_BACKUP = THREAD_ID
THREAD_ID = TIMEOUT_THREAD_ID
try:
    initial = {
        "question": RESEARCH_QUESTION,
        "doc_ids": [], "docs": [], "summaries": [], "answer": "",
        "run_label": "timeout",
    }
    timeout_result = graph.invoke(
        initial,
        config={"configurable": {"thread_id": TIMEOUT_THREAD_ID}},
    )
    print(timeout_result["answer"])
finally:
    _anthropic_client.messages.create = _real_messages_create
    THREAD_ID = THREAD_ID_BACKUP
```

</details>

**Wallet check.** Crash 3 of 3 recovered. Tool calls saved: 1 (the LLM retry caught the timeout inside the summarize node). Spent so far: about $0.90 across four runs.

## Task 5: SQL aggregate proves no re-execution across the three recoveries

Aggregate the runs table by `thread_id` and sum `tool_calls_in_step`. Per-thread totals should fall inside the expected 5 to 7 range. A thread with 10+ tool calls means the agent re-executed steps that the checkpointer should have skipped. A thread under 4 means the resume never finished.

In [ ]:
# NBVAL_SKIP
# Task 5: run the aggregate and print the per-thread breakdown.

# YOUR CODE: query the runs table via supabase.table(RUNS_TABLE)
# YOUR CODE:   .select("thread_id,tool_calls_in_step").execute()
# YOUR CODE: aggregate by thread_id: sum(tool_calls_in_step)
# YOUR CODE: print each thread_id and its total
# YOUR CODE: store the per-thread totals in a dict named `per_thread_calls`

per_thread_calls = {}

In [ ]:
# NBVAL_SKIP
# Checkpoint 5: per-thread tool call totals fall inside the 4-to-7 range,
# none of them in the 10+ runaway-re-execution range.


def checkpoint_5(session):
    if not isinstance(per_thread_calls, dict) or not per_thread_calls:
        return CheckpointResult(
            status="fail",
            error_reason="per_thread_calls is empty; the Task 5 aggregate did not run.",
        )
    bad = {tid: n for tid, n in per_thread_calls.items() if n > 7 or n < 4}
    if bad:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Per-thread tool call totals outside the expected 4 to 7 range: {bad}. "
                f"A thread with 10+ means the retry was at the graph level and re-executed prior nodes. "
                f"A thread with under 4 means the resume never completed."
            ),
        )
    return CheckpointResult(status="pass")


check(5, checkpoint_5)

<details><summary>Hint 1 (nudge)</summary>

Supabase Python does not expose `GROUP BY` on its REST builder. Pull the rows down and aggregate in Python; it is cheaper than a roundtrip and the dataset is small.

</details>

<details><summary>Hint 2 (direction)</summary>

```
resp = supabase.table(RUNS_TABLE).select("thread_id,tool_calls_in_step").execute()
per_thread_calls = {}
for row in resp.data:
    per_thread_calls[row["thread_id"]] = per_thread_calls.get(row["thread_id"], 0) + row["tool_calls_in_step"]
for tid, n in per_thread_calls.items():
    print(f"  {tid}: {n} tool calls")
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 5:

```python
resp = supabase.table(RUNS_TABLE).select("thread_id,tool_calls_in_step").execute()
per_thread_calls = {}
for row in resp.data:
    per_thread_calls[row["thread_id"]] = (
        per_thread_calls.get(row["thread_id"], 0) + row["tool_calls_in_step"]
    )
for tid, n in sorted(per_thread_calls.items()):
    print(f"  {tid}: {n} tool calls")
```

</details>

## Cleanup

Drop the Supabase tables (your bookkeeping plus LangGraph's checkpointer schema) and remove the local trace file. The verification scan re-queries `information_schema.tables` to confirm every name is gone; a stale orphan triggers the dirty-cleanup exit.

In [ ]:
# NBVAL_SKIP
# Cleanup. Drop Supabase tables in reverse-creation order, plus the
# LangGraph checkpointer tables (created implicitly by PostgresSaver.setup),
# plus the local trace file.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Supabase project may still hold lab tables. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: about $2.00.** One clean run plus one crash-recovery resume plus one blip-recovery fresh run plus one timeout-recovery fresh run. Roughly 50K tokens across Sonnet and Haiku. Supabase Postgres held the checkpoint state for free. Cleanup dropped every Supabase table this lab created plus LangGraph's checkpointer schema, removed the local trace file, and the Supabase quota is restored.

Checkpoints dropped. Runs table gone. Supabase quota restored.

## Reflection

These are not graded. They are for you.

1. You killed your kernel and the agent resumed. In production you would not be killing the Python process; you would have a deployment rollover or a pod eviction. What is the production equivalent of "kernel restart" for this agent, and what would change about the checkpointer config? Walk through what `PostgresSaver`'s schema needs to look like when 50 worker pods all hold concurrent agent runs against the same Postgres instance.

2. The Postgres checkpointer commits after every node. Imagine the team's traffic is 10K agent runs per day, six nodes each, peak roughly 5x the average. What is the rough write QPS on Postgres at peak, and what is one optimization you would try first if the DB becomes a bottleneck? Connection pooling, table partitioning, write batching, all of the above, or something else?

## Exam-Style Practice

**Question 1 (durable workflow semantics):**

A team's agent has a five-step workflow that crashes between steps 3 and 4. They want "exactly-once" execution semantics for each tool call (the side effects of a tool should not happen twice if the same tool is retried). Which is necessary?

A. Idempotent tool implementations.
B. A larger LLM at the planner step.
C. A faster checkpointer (in-memory instead of Postgres).
D. Synchronous tool calls only.

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: the checkpointer guarantees at-least-once resume of any step that crashed mid-execution. The end-to-end semantics get to exactly-once only if the tools themselves are idempotent (calling them twice is the same as calling them once).
- B is wrong: model size is unrelated to delivery semantics.
- C is wrong: in-memory checkpointers are FASTER but they lose state on crash. The whole point of the Postgres checkpointer is durability; speed is a separate axis.
- D is wrong: synchronous vs async does not change retry semantics; it changes concurrency.

</details>

**Question 2 (retry boundary):**

An agent's `summarize` node calls an LLM that occasionally times out. The team adds a retry. The wrong implementation re-runs every prior node on each retry and burns 5x the tokens. Which retry placement preserves the checkpointer's accumulated state?

A. Retry inside the tool/LLM call (catch the exception in the node body, retry just the failing call).
B. Retry at the graph level (catch the exception in the `invoke` caller, re-invoke the graph from the start).
C. Retry only on the next user request (let the user re-send the question).
D. Retry by spawning a fresh thread_id and discarding the partial state.

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: the node-level retry catches the transient error inside the failing node, re-runs only the failing call, and the checkpointer's prior-node state is intact. Token cost stays bounded.
- B is wrong: the graph-level retry is exactly the 5x-tokens anti-pattern the question describes.
- C is wrong: punting to the user breaks the durable-workflow contract entirely.
- D is wrong: a fresh thread_id starts from scratch and pays for every step again; this is identical to the naive non-durable agent.

</details>